 Exploratory Data Analysis and Visualization Assignment-1.
 
 Roll num:160123733002
 
 Name: A.Shriya Reddy
 
 Titanic Dataset
 
 Link:https://www.kaggle.com/datasets/yasserh/titanic-dataset


In [34]:
import pandas as pd
import numpy as np
df = pd.read_csv("Titanic-Dataset.csv")
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [35]:
'''Q1 Create a Series from the Age; column. Examine the distribution
of passenger ages using descriptive statistics and
identify the age group with the highest fatality rate.
Justify your observations.'''

age_series = df['Age']
# Descriptive statistics
age_stats = age_series.describe()
print(age_stats)
bins = [0, 12, 19, 35, 60, 100]
labels = ['Child', 'Teen', 'Young Adult', 'Adult', 'Senior']
df['AgeGroup'] = pd.cut(df['Age'], bins=bins, labels=labels)
# Fatality rate per age group
fatality_rate = df[df['Survived'] == 0].groupby('AgeGroup')['Survived'].count() / df.groupby('AgeGroup')['Survived'].count()
print(fatality_rate)

count    714.000000
mean      29.699118
std       14.526497
min        0.420000
25%       20.125000
50%       28.000000
75%       38.000000
max       80.000000
Name: Age, dtype: float64
AgeGroup
Child          0.420290
Teen           0.589474
Young Adult    0.615616
Adult          0.600000
Senior         0.772727
Name: Survived, dtype: float64


1 ans : The age distribution shows most passengers fall between 20–40 years.
Fatality rate is highest among senior passengers (77%), indicating vulnerability.
Children have lower fatality rates due to “women and children first” rescue policy.
Hence, age significantly influenced survival chances.

In [26]:
'''Q2 Using the DataFrame, determine which
passenger class (Pclass) had the best survival rate.
Apply boolean indexing and conditional selection to
filter survivors and non-survivors, then compare
results across genders.'''

survival_by_class = df.groupby('Pclass')['Survived'].mean()
# Boolean indexing
survivors = df[df['Survived'] == 1]
non_survivors = df[df['Survived'] == 0]
gender_survival = df.groupby(['Pclass', 'Sex'])['Survived'].mean()
print(survival_by_class)
print(gender_survival)

Pclass
1    0.629630
2    0.472826
3    0.242363
Name: Survived, dtype: float64
Pclass  Sex   
1       female    0.968085
        male      0.368852
2       female    0.921053
        male      0.157407
3       female    0.500000
        male      0.135447
Name: Survived, dtype: float64


2 ans: First-class passengers had the highest survival rate (~63%), followed by second and third class.
Females had significantly higher survival rates across all classes.
This reflects social hierarchy and priority-based evacuation.
Therefore, both class and gender strongly influenced survival.

In [37]:
''' Q3 Identify all missing values in the dataset.
Compare the effect of replacing missing &#39;Age&#39; values
with the mean versus the median. Which strategy is
more appropriate and why? Support your answer with
statistical evidence.'''


missing = df.isnull().sum()
print(missing)
mean_age = df['Age'].mean()
median_age = df['Age'].median()
df_mean = df.copy()
df_median = df.copy()
df_mean['Age'] = df_mean['Age'].fillna(mean_age)
df_median['Age'] = df_median['Age'].fillna(median_age)
print("Mean:", mean_age)
print("Median:", median_age)

PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
AgeGroup       177
dtype: int64
Mean: 29.69911764705882
Median: 28.0


3 ans: The Age column contains 177 missing values.
Mean (29.7) is higher than median (28), indicating a right-skewed distribution.
Median is less affected by outliers compared to mean.
Hence, median imputation is more appropriate.

In [36]:
''' Q4 Apply hierarchical indexing on the Titanic
DataFrame using &#39;Pclass&#39; and &#39;Sex&#39; as a multi-level
index. Examine survival counts at each hierarchical
level using xs() and loc[].'''


multi_df = df.set_index(['Pclass', 'Sex']).sort_index()
# Access using loc
print(multi_df.loc[(1, 'female')].head())
# Access using xs
print(multi_df.xs('male', level='Sex').head())
survival_counts = df.groupby(['Pclass', 'Sex'])['Survived'].sum()
print(survival_counts)

               PassengerId  Survived  \
Pclass Sex                             
1      female            2         1   
       female            4         1   
       female           12         1   
       female           32         1   
       female           53         1   

                                                            Name   Age  SibSp  \
Pclass Sex                                                                      
1      female  Cumings, Mrs. John Bradley (Florence Briggs Th...  38.0      1   
       female       Futrelle, Mrs. Jacques Heath (Lily May Peel)  35.0      1   
       female                           Bonnell, Miss. Elizabeth  58.0      0   
       female     Spencer, Mrs. William Augustus (Marie Eugenie)   NaN      1   
       female           Harper, Mrs. Henry Sleeper (Myna Haxtun)  49.0      1   

               Parch    Ticket      Fare Cabin Embarked     AgeGroup  
Pclass Sex                                                            
1      fe

4 ans: Hierarchical indexing using Pclass and Sex allows structured analysis.
Female passengers in higher classes show higher survival counts.
MultiIndex simplifies accessing multi-dimensional grouped data.
Sorting improves performance and avoids indexing warnings.

In [38]:
'''Q5 Create a new derived column &#39;FamilySize&#39; =
SibSp + Parch + 1. Using ufuncs, compute a
&#39;SurvivalScore&#39; that weighs Fare and FamilySize.
Construct a final ranked DataFrame sorted by
SurvivalScore.'''

df['FamilySize'] = df['SibSp'] + df['Parch'] + 1
df['SurvivalScore'] = np.log1p(df['Fare']) * df['FamilySize']
ranked_df = df.sort_values(by='SurvivalScore', ascending=False)

ranked_df[['Fare', 'FamilySize', 'SurvivalScore']].head()

,Fare,FamilySize,SurvivalScore
180,69.55,11,46.819538
324,69.55,11,46.819538
159,69.55,11,46.819538
201,69.55,11,46.819538
863,69.55,11,46.819538


5 ans: Higher fare + larger family size may indicate better survival chances.
Log transformation avoids skewness.

In [39]:
'''Q6 Perform index alignment between two Series: one
containing average fare per Pclass, and another
containing average age per Pclass. Examine how
Pandas aligns indices during arithmetic operations.'''

fare_series = df.groupby('Pclass')['Fare'].mean()
age_series = df.groupby('Pclass')['Age'].mean()
# Alignment
result = fare_series + age_series
print(result)

Pclass
1    122.388128
2     50.539813
3     38.816170
dtype: float64


6 ans:Pandas automatically aligns data based on index labels during operations.
Both Series share Pclass as index, ensuring correct addition.
This avoids mismatched calculations.
Index alignment is a key feature for accurate data operations.

In [30]:
'''Q7 Examine the relationship between &#39;Embarked&#39;
port and survival rate using groupby with hierarchical
indexing. Assess whether port of embarkation
significantly influenced survival outcomes.'''

embarked_survival = df.groupby(['Embarked'])['Survived'].mean()
print(embarked_survival)

Embarked
C    0.553571
Q    0.389610
S    0.336957
Name: Survived, dtype: float64


7 ans: Passengers from port C have the highest survival rate (~55%).
Ports Q and S show lower survival rates.
However, differences are not very large.
Hence, Embarked location had limited influence compared to class and gender.

In [31]:
'''Q8 Design a function using apply() and map() to
categorize passengers into &#39;Child&#39;, &#39;Adult&#39;, and &#39;Senior&#39;
based on age. Integrate this with a pivot table showing
survival rates per age category and gender.'''

def categorize(age):
    if age < 18:
        return 'Child'
    elif age < 60:
        return 'Adult'
    else:
        return 'Senior'

df['Category'] = df['Age'].apply(categorize)

pivot = pd.pivot_table(df, values='Survived',
                       index='Category',
                       columns='Sex',
                       aggfunc='mean')

print(pivot)

Sex         female      male
Category                    
Adult     0.767327  0.179625
Child     0.690909  0.396552
Senior    0.701754  0.130137


8 ans: Passengers were categorized into Child, Adult, and Senior groups.

In [40]:
'''Q9 Examine null value patterns across all columns.
Using isnull(), notnull(), and dropna() with different
thresh and axis parameters, determine the optimal
strategy for cleaning the dataset.'''

# Null checks
print(df.isnull().sum())
# Drop with threshold
df_clean = df.dropna(thresh=5)
# Drop columns with many nulls
df_clean2 = df.dropna(axis=1)
df_clean.head()

PassengerId        0
Survived           0
Pclass             0
Name               0
Sex                0
Age              177
SibSp              0
Parch              0
Ticket             0
Fare               0
Cabin            687
Embarked           2
AgeGroup         177
FamilySize         0
SurvivalScore      0
dtype: int64


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,AgeGroup,FamilySize,SurvivalScore
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S,Young Adult,2,4.220426
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C,Adult,2,8.561186
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S,Young Adult,1,2.188856
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S,Young Adult,2,7.981668
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S,Young Adult,1,2.202765


9 ans:
Cabin column has a large number of missing values (687), making it unreliable.
Age has moderate missing values and should be imputed.
Embarked has very few missing values and can be filled or dropped.
Best strategy:
Drop highly missing columns (Cabin)
Fill important columns (Age)

In [41]:
'''Q10 Create a comprehensive summary DataFrame
using multi-level groupby combining Pclass, Sex, and
Survived. Use stack() and unstack() to reshape the
result into a cross-tabulation report.'''

summary = df.groupby(['Pclass', 'Sex', 'Survived']).size()
report = summary.unstack().fillna(0)
print(report)

Survived         0   1
Pclass Sex            
1      female    3  91
       male     77  45
2      female    6  70
       male     91  17
3      female   72  72
       male    300  47


10 ans: Multi-level grouping provides detailed survival distribution.
Reshaping using unstack() improves readability and creates a clear report


CONCLUSION
The exploratory data analysis of the Titanic dataset provided meaningful insights into factors influencing passenger survival. Through the use of Pandas operations such as indexing, grouping, hierarchical indexing, and handling missing data, key patterns were identified.